# Generating simulation datasets

This self-contained tutorial generates five Q2237 image-B-like realizations, followed by five systems drawn from explicit lens and disk priors. All systems, sources, priors, and simulation calls are defined here. No generation script or saved simulation is loaded.

The CUDA calculation uses $N=10^7$, $k=2$, $r=2$, $v=4$, the Taylor far-field approximation, frame-zero $k=1\rightarrow2$ normalization, temporal batch 30, and ten-frame endpoint-union scout refreshes. Maps and source-center labels are evaluated every 25 days for ten years, while source brightness is evaluated daily. The memory-intensive 8192-square labeled jobs run one curve at a time. Stellar motions include the random stellar dispersion, sampled lens and source peculiar velocities, and the CMB dipole projected at each system sky position. The CPU settings are a smaller numerical demonstration, not a production timing comparison.

Each source is a full-Kerr thin disk whose one-day stochastic driver heats an axial lamppost and produces a wavelength-dependent delayed thermal response. With `include_microlensing_only=True`, the same call also evaluates the disk at its mean driver. The two products share one map and label stream, and the static mean-driver brightness is evaluated only once; thermal reprocessing cannot be removed by dividing by the driver amplitude. These light curves require the disk's static Kerr observer coordinates and lamppost illumination profile, but do not request the optional binned reverberation transfer-function data product. Both curves use physical Jy fluxes, which can be converted to magnitudes.

The first call can include compilation. Compatible later calls reuse kernels, although changing a numerical shape or stellar-count specialization can require additional compilation. Timing below includes maps, photometry, and center labels, but excludes physical setup and the separate map galleries.


In [ ]:
from pathlib import Path
import numpy as np
import torch
from matplotlib import pyplot as plt
import microcaustics as mc
import microcaustics.plotting as mcp

plt.rcParams.update(mcp.publication_style())
print(mcp.runtime_description())
capabilities = mc.RuntimeCapabilities.detect()
use_cuda = capabilities.cuda_available and capabilities.triton_importable
runtime = mc.RuntimeConfig(
    device="cuda" if use_cuda else "cpu",
    backend="triton" if use_cuda else "torch-eager",
    strict_backend=use_cuda, dtype=torch.float32, memory_fraction=0.95,
)
OUTPUT = Path("generated/simulation_datasets")
OUTPUT.mkdir(parents=True, exist_ok=True)

# Main accuracy and throughput controls. Change these for your application.
count = 5
duration_days = 3650.0
map_cadence_days = 25.0 if use_cuda else duration_days / 12
source_cadence_days = 1.0
rays = 10_000_000 if use_cuda else 262_144
source_pixels = 1024 if use_cuda else 128
label_pixels = 8192 if use_cuda else 512
temporal_batch_size = 30
label_batch_size = 10 if use_cuda else 1
source_setup_batch_size = 3 if use_cuda else 1
labeled_curves_per_batch = 1
scout_refresh_frames = 10
bands = {"u": 3671, "g": 4827, "r": 6223, "i": 7546, "z": 8691, "y": 9712}


In [ ]:
def make_system(
    seed, *, lens_redshift=0.0395, source_redshift=1.695,
    convergence=0.391, shear=0.391, shear_angle_deg=141.73,
    smooth_matter_fraction=0.0, black_hole_mass_solar=10.0**9.08,
    eddington_ratio=0.34, spin=0.74, inclination_deg=10.0,
    position_angle_deg=0.0, ra_deg=340.126125, dec_deg=3.358611,
):
    """Define one physical system with automatic source and stellar apertures."""
    driver = mc.broken_power_law_driving_signal(
        cadence_days=1.0, break_timescale_days=200.0,
        alpha_L=1.0, alpha_R=3.0, standard_deviation=0.10,
    )  # inherits an independent variability stream from the system seed
    disk = mc.KerrDiskModel(
        black_hole_mass_solar=black_hole_mass_solar,
        eddington_ratio=eddington_ratio, bands_angstrom=bands,
        spin=spin, inclination_deg=inclination_deg, position_angle_deg=position_angle_deg,
        source_redshift=source_redshift,
        lamp_fraction=0.1, corona_height_above_isco_rg=20.0,
        driving_signal=driver, compile_solver=use_cuda,
        source_grid_shape=source_pixels, enclosed_flux_fraction=0.999, source_margin=1.05,
    )
    return mc.MicrolensingSystem(
        macro=mc.MacroLens(
            convergence=convergence, shear=shear, shear_angle_deg=shear_angle_deg,
            smooth_matter_fraction=smooth_matter_fraction,
        ),
        lens_redshift=lens_redshift, source_redshift=source_redshift,
        source=disk,
        stellar_population=mc.StellarPopulation.salpeter(
            mean_mass_solar=0.3, mass_ratio=100.0,
            kinematics=mc.SkyProjectedKinematics(
                ra_deg=ra_deg, dec_deg=dec_deg,
                stellar_dispersion_km_s=170.0,
                peculiar_velocity_dispersion_km_s=235.0,
                include_cmb_dipole=True, seed=seed,
            ),
        ),
        seed=seed, duration_days=duration_days, light_loss=0.01, safety_scale=1.5,
        caustic_grid_shape=label_pixels, runtime=runtime,
    )


systems = [make_system(seed) for seed in range(count)]  # seeds 0 through 4


In [ ]:
def generate_dataset(systems):
    """Batch driven and microlensing-only curves through shared maps."""
    center_samples = [[] for _ in systems]

    def center_observer(samples):
        def capture_center(index, frame):
            values = frame.magnification_map.values
            samples.append(
                values[values.shape[0] // 2, values.shape[1] // 2].detach().clone()
            )
        return capture_center

    observers = [center_observer(samples) for samples in center_samples]
    batch = mc.batched_system_light_curves(
        systems, duration_days=duration_days,
        map_cadence_days=map_cadence_days,
        source_cadence_days=source_cadence_days,
        curves_per_batch=labeled_curves_per_batch, include_labels=True,
        include_microlensing_only=True, rays=rays,
        source_setup_batch_size=source_setup_batch_size,
        temporal_batch_size=temporal_batch_size,
        label_batch_size=label_batch_size,
        scout_refresh_frames=scout_refresh_frames,
        map_observers=observers, profile=True,
    )
    curves = list(batch.light_curves)
    microlensing_fluxes = [curve.microlensing_only_flux for curve in curves]
    centers = [torch.stack(samples).cpu() for samples in center_samples]
    print("Requested labeled curves per batch", batch.requested_curves_per_batch)
    print("Executed batch sizes", batch.executed_batch_sizes)
    print("OOM reductions", batch.oom_reductions)
    print(f"Driven amortized wall time [s/curve] {batch.seconds_per_curve:.3f}")
    print("Microlensing-only comparison shares the same map and label stream")
    print(f"Map/label epochs {curves[0].labels.times_days.numel()}, photometry epochs {curves[0].times_days.numel()}")
    return curves, microlensing_fluxes, centers, batch


curves, microlensing_only_fluxes, centers, batch = generate_dataset(systems)
figure, axes = mcp.plot_light_curve_dataset(
    curves, center_magnifications=centers,
    microlensing_fluxes=microlensing_only_fluxes, band="i",
)
figure.savefig(OUTPUT / "five_q2237_b_training_curves.pdf", bbox_inches="tight")
plt.show()


## Inspect one magnification map from every realization

The same in-memory systems now generate a $t=0$ map and its caustics. These visualization calls are outside the light-curve timing. The full ten-year stellar apertures remain unchanged. Each map uses the dynamic $k=2$ method with its frame-zero normalization correction.

The microlensing-only product preserves the steady viscous and lamppost heating while removing only stochastic driving. It is attached to each returned curve as `microlensing_only_flux` and `microlensing_only_magnitude`.

The three diagnostics are the point magnification at the map center, the center crossing flag, and the center distance to the nearest caustic. Distance is capped at the inscribed source radius when the in-field caustic search cannot determine a larger distance. No extra markers or transformed distance axis are used.


In [ ]:
def map_gallery(systems):
    """Generate one labeled map per existing system, separately from LC timing."""
    method = mc.production_ipm_config(rays=rays)
    frames = [
        next(iter(system.dynamic_labeled_maps(
            [0.0], method=method,
            schedule=mc.production_dynamic_config(temporal_batch_size=1),
        )))
        for system in systems
    ]
    return mcp.plot_labeled_map_gallery(frames)


figure, axes = map_gallery(systems)
figure.savefig(OUTPUT / "five_q2237_b_maps.pdf", bbox_inches="tight")
plt.show()


For multiple GPUs, run `python examples/generate_q2237_training_set.py --count 1000 --seed 0 --gpus 0 1 2 3 --curves-per-batch 3` from a repository checkout. Each worker remains on one GPU and reuses compatible kernels. The script also saves compact records and can resume completed files. It is not required to execute this notebook.


## Prior-sampled simulation datasets

Next we use the same system constructor and simulation calls with explicitly sampled physical parameters.


## Training sets from explicit lens and disk priors

The next five systems use the demonstration priors below. They are not a population model. A scientific inference run should supply survey- and selection-aware priors.

Each system gets its own source, stellar field, and intrinsic driver. Sources are evaluated daily and maps and center labels use the coarser map cadence. Only numerical shapes are kept fixed to encourage compiled-kernel reuse.

The prior RNG starts at seed 0. Each system also receives its example index as its seed, from which the package derives separate stellar and variability streams. Change either seed deliberately when generating a new dataset. The sampled parameter dictionaries stay in memory alongside the systems and results.


## Demonstration priors and their limits

These bounds were deliberately chosen as broad, transparent *demonstration priors* spanning ordinary galaxy-scale quasar macroimages and thin disks. They were not inferred from a lens survey and should not be interpreted as a population distribution. The draws are independent except for $z_s>z_l+0.4$ and the macro-Jacobian guard described below. For a scientific training set, replace them with a joint, selection-aware prior or posterior samples for the systems being modeled.

| Quantity | Sampling distribution | Demonstration rationale |
|---|---|---|
| Lens redshift $z_l$ | $\mathcal U(0.15,0.75)$ | Broad galaxy-lens redshift interval |
| Source redshift $z_s$ | $\mathcal U(\max[1,z_l+0.4],3)$ | Background quasars with clear lens--source separation |
| Convergence $\kappa$ | $\mathcal U(0.20,0.65)$ | Spans common local macroimage environments |
| Shear $\gamma$ | $\mathcal U(0.10,0.60)$ | Spans weak through strong local tidal fields |
| Smooth-matter fraction $s$ | $\mathcal U(0,0.5)$ | Samples all-stellar through mixed smooth/compact surface density |
| Shear angle | $\mathcal U(0,180^\circ)$ | Isotropic lens-plane orientation |
| Sky position | uniform on the celestial sphere | Sets the projected CMB contribution to bulk motion |
| $\log_{10}(M_{\rm BH}/M_\odot)$ | $\mathcal U(7.5,9.5)$ | Broad quasar black-hole mass interval |
| Eddington ratio | log-uniform on $[10^{-1.3},10^{-0.05}]$ | Covers sub-Eddington thin disks over a wide luminosity range |
| Spin $a$ | $\mathcal U(-0.8,0.95)$ | Broad retrograde-to-prograde interval without extremal endpoints |
| Inclination $i$ | uniform in $\cos i$ for $0\le i\le60^\circ$ | Isotropic type-1 orientations with more edge-on systems excluded |
| Disk position angle | $\mathcal U(0,180^\circ)$ | Isotropic projected orientation |

Draws are rejected when either absolute macro-Jacobian eigenvalue, $|1-\kappa\pm\gamma|$, is below 0.12. This keeps a compact tutorial away from nearly singular macro configurations. Compact objects follow a Salpeter mass function with mass ratio 100 and mean mass $0.3\,M_\odot$. Their positions are uniform in a circular lens field, and the compact convergence is $(1-s)\kappa$. The velocity model includes a 170 km s$^{-1}$ stellar dispersion, independently sampled lens and source peculiar velocities with a 235 km s$^{-1}$ normalization, and the CMB dipole projected at the sampled sky position. The source field encloses 99.9% of the reddest-band thin-disk flux and adds a 5% margin. The circular lens-field diameter is 1.5 times the diagonal of the 1%-light-loss rectangular estimate.

A fixed ray count is a compute budget, not a universal accuracy guarantee. Changing redshift, macro magnification, lens-field size, and disk angular size changes how many traced polygons resolve each source pixel. The system summaries record these quantities so a production data generator can enforce a convergence criterion, increase `rays`, or adjust source resolution for difficult draws.

In [ ]:
def sample_parameters(rng):
    """Draw the documented demonstration prior, excluding near-singular macroimages."""
    lens_redshift = rng.uniform(0.15, 0.75)
    source_redshift = rng.uniform(max(1.0, lens_redshift + 0.4), 3.0)
    for _ in range(10_000):
        convergence = rng.uniform(0.20, 0.65)
        shear = rng.uniform(0.10, 0.60)
        if min(abs(1 - convergence - shear), abs(1 - convergence + shear)) > 0.12:
            break
    else:
        raise RuntimeError("Could not draw a nonsingular macroimage from these priors")
    return {
        "lens_redshift": lens_redshift, "source_redshift": source_redshift,
        "convergence": convergence, "shear": shear,
        "smooth_matter_fraction": rng.uniform(0.0, 0.5),
        "shear_angle_deg": rng.uniform(0.0, 180.0),
        "black_hole_mass_solar": 10.0**rng.uniform(7.5, 9.5),
        "eddington_ratio": 10.0**rng.uniform(-1.3, -0.05),
        "spin": rng.uniform(-0.8, 0.95),
        "inclination_deg": np.degrees(np.arccos(rng.uniform(0.5, 1.0))),
        "position_angle_deg": rng.uniform(0.0, 180.0),
        "ra_deg": rng.uniform(0.0, 360.0),
        "dec_deg": np.degrees(np.arcsin(rng.uniform(-1.0, 1.0))),
    }


rng = np.random.default_rng(0)
parameters = [sample_parameters(rng) for _ in range(count)]


In [ ]:
random_systems = [
    make_system(seed, **values) for seed, values in enumerate(parameters)
]
random_curves, random_microlensing_fluxes, random_centers, random_batch = generate_dataset(random_systems)


In [ ]:
annotations = [
    rf'$\kappa={p["convergence"]:.2f},\ \gamma={p["shear"]:.2f},\ '
    rf'\log M_{{\rm BH}}={np.log10(p["black_hole_mass_solar"]):.2f}$'
    for p in parameters
]
figure, axes = mcp.plot_light_curve_dataset(
    random_curves, center_magnifications=random_centers,
    microlensing_fluxes=random_microlensing_fluxes,
    band="i", annotations=annotations,
)
figure.savefig(OUTPUT / "five_prior_sampled_training_curves.pdf", bbox_inches="tight")
plt.show()


## Inspect the prior-sampled maps

Generate one $t=0$ map for each of the same five systems. Their source sizes and stellar apertures differ physically, while the plots share a magnification color scale. These calls are again outside the reported light-curve timing.


In [ ]:
figure, axes = map_gallery(random_systems)
figure.savefig(OUTPUT / "five_prior_sampled_maps.pdf", bbox_inches="tight")
plt.show()


For multiple GPUs, use `python examples/generate_random_training_set.py --count 1000 --seed 0 --gpus 0 1 2 3 --curves-per-batch 3`. That script provides its own parameter records and restartable output files. It is an alternative to this in-memory workflow.

## Batching independent systems

The two complete datasets above are generated directly with `mc.batched_system_light_curves`. The function accepts any mixture of independent single-, double-, quad-, or other multi-image systems. It batches individual macroimage curves on one GPU and reconstructs the original system grouping and image order. Each image retains its own lens, stars, trajectory, and map sequence, while a multi-image system retains its shared intrinsic source and arrival delays. This differs from shared-map source batching, which evaluates several sources behind the same map sequence.

Choose `curves_per_batch` explicitly; it counts individual macroimage light curves rather than systems. `source_setup_batch_size` independently pools compatible driven Kerr source setup, with one shared disk per multi-image system. Lossless CUDA OOM backoff reduces either concurrency without lowering numerical accuracy. For large datasets, pass a directory to `output_path` for flat per-image NPZ files plus a manifest, or a `.npz` path for one combined archive, then use `batch.load_system(index)`. A bounded writer overlaps saving with GPU work. For a new accelerator or a different stellar population, optional `mc.tune_system_light_curve_batch` can compare candidate curve concurrency values. A larger batch is not necessarily faster.


### Seeded Rubin WFD observations from the batched dataset

Build the OpSim index once, select one reproducible random WFD field, and apply that same six-band cadence to every simulated system. Observation-noise seeds remain independent of the seeds that generated the stars and intrinsic variability. The database is intentionally external to the package; set `MICROCAUSTICS_LSST_OPSIM` or place `baseline_v4.3.5_10yrs.db` in a parent directory.

The plotted example shows all six Rubin bands from the first batched light curve.


In [ ]:
opsim_path = mc.find_rubin_opsim_database()
if opsim_path is None:
    raise FileNotFoundError(
        "Set MICROCAUSTICS_LSST_OPSIM to a Rubin OpSim SQLite database "
        "or place baseline_v4.3.5_10yrs.db in a parent directory."
    )
wfd_seed = 2026
rubin = mc.RubinOpSimCadenceIndex.from_database(opsim_path)
cadence = rubin.sample(seed=wfd_seed, survey="wfd").select_bands(tuple(bands))
assert set(cadence.band_names) == set(bands)
observed_dataset = tuple(
    mc.observe_light_curve(curve, cadence, image_name=f"system_{index:04d}", seed=10_000 + index)
    for index, curve in enumerate(curves)
)
preview_curves = mc.MultiImageLightCurves((
    mc.MacroImageLightCurve("system_0000", 0.0, curves[0]),
))
preview_observations = mc.observe_multi_image_light_curves(
    preview_curves, cadence, seed=10_000,
)
print("Using Rubin OpSim database:", opsim_path)
print("WFD metadata:", cadence.metadata)
print("Visits by band:", {name: cadence.band_names.count(name) for name in bands})
figure, axis = mcp.plot_photometric_observations(
    preview_observations, image="system_0000", marker_size=4.0, capsize=2.0,
)
figure.savefig(OUTPUT / "seeded_wfd_six_band_observations.pdf", bbox_inches="tight")
plt.show()
